# Prediccion de Zona de Destino -- NYC Yellow Taxi 2015-01

**Tarea 1 -- Sistemas Urbanos Inteligentes (ICT3115, 2026-1)**  
**Problema:** Clasificacion multiclase -- predecir la zona de destino (`DOLocationID`, hasta 265 clases)  
**Features:** tiempo de pickup + zona de origen (PULocationID) + metadata Census ACS del barrio de origen  
**Autores:** Nicolas Herrera y Vincent Metzker

> **Nota:** todas las features usadas son informacion disponible en el momento del pickup.

Este notebook tiene las siguientes secciones:
1. Carga de datos y preparacion
2. Analisis del target variable
3. Split y preprocesamiento
4. MLP base (sin embeddings)
5. MLP con embeddings (PULocationID como embedding)
6. Analisis de sobreajuste + regularizacion
7. AutoEncoder + MLP
8. Analisis critico final

## 0. Setup

In [ ]:
import random, math, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Dataset

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             classification_report, top_k_accuracy_score)
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid', palette='muted')

In [ ]:
SEED = 22041991

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed()

if torch.cuda.is_available():            DEVICE = 'cuda'
elif torch.backends.mps.is_available():  DEVICE = 'mps'
else:                                    DEVICE = 'cpu'

print('Dispositivo:', DEVICE)
DATA_DIR = '../data'

## 1. Carga y preparacion de datos

### 1.1 Cargar muestra

In [ ]:
df = pd.read_parquet(os.path.join(DATA_DIR, 'trips_sample_with_cats.parquet'))
print(f'Shape cargado: {df.shape}')
df.head(3)

### 1.2 Agregar metadata Census si faltan

Si el parquet fue guardado antes de correr la seccion 4.7 del notebook de EDA,
los campos census y borough no estaran presentes -- este bloque los anade.

In [ ]:
if 'PU_Borough_id' not in df.columns:
    print('Columnas census no encontradas -- ejecutando join...')
    lookup = pd.read_csv(os.path.join(DATA_DIR, 'taxi_zone_lookup_enriched.csv'))
    drop_cols = ['Zone', 'service_zone']
    pu_lookup = (lookup.drop(columns=drop_cols, errors='ignore')
                 .add_prefix('PU_').rename(columns={'PU_LocationID': 'PULocationID'}))
    do_lookup = (lookup.drop(columns=drop_cols, errors='ignore')
                 .add_prefix('DO_').rename(columns={'DO_LocationID': 'DOLocationID'}))
    df = df.merge(pu_lookup, on='PULocationID', how='left')
    df = df.merge(do_lookup, on='DOLocationID', how='left')
    BOROUGH_MAP = {'Manhattan': 0, 'Brooklyn': 1, 'Queens': 2,
                   'Bronx': 3, 'Staten Island': 4, 'EWR': 5}
    df['PU_Borough_id'] = df['PU_Borough'].map(BOROUGH_MAP).fillna(6).astype(np.int8)
    df['DO_Borough_id'] = df['DO_Borough'].map(BOROUGH_MAP).fillna(6).astype(np.int8)
    df.drop(columns=['PU_Borough', 'DO_Borough'], inplace=True, errors='ignore')
    census_cols = [c for c in df.columns if c.startswith(('PU_', 'DO_'))
                   and c not in ('PU_Borough_id', 'DO_Borough_id')]
    df[census_cols] = df[census_cols].fillna(0)
    print('Join completado:', df.shape)
else:
    print('Columnas census ya presentes.')

### 1.3 Definir target y features

In [ ]:
TARGET = 'DOLocationID'

# Filtrar zonas raras (menos de MIN_SAMPLES viajes)
MIN_SAMPLES = 50
df['do_zone_idx'] = (df[TARGET] - 1).clip(0, 264).astype(np.int64)
zone_counts = df['do_zone_idx'].value_counts()
valid_zones  = zone_counts[zone_counts >= MIN_SAMPLES].index
df = df[df['do_zone_idx'].isin(valid_zones)].copy()

# Reindexar a indices consecutivos 0, 1, 2, ...
zone_remap = {old: new for new, old in enumerate(sorted(valid_zones))}
df['do_zone_idx'] = df['do_zone_idx'].map(zone_remap)
remap_inv  = {v: k+1 for k, v in zone_remap.items()}  # idx -> LocationID original
N_CLASSES  = len(valid_zones)

# Lookup de nombres de zona
zone_lookup = pd.read_csv(os.path.join(DATA_DIR, 'taxi_zone_lookup_enriched.csv'),
                           usecols=['LocationID', 'Zone', 'Borough'])
zone_lookup = zone_lookup.set_index('LocationID')

# Features numericas (disponibles en tiempo de pickup)
NUM_FEATS = [
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
    'PU_Population', 'PU_MedianHouseholdIncome', 'PU_HousingUnits',
    'PU_Transport_CarAlone_pct', 'PU_Transport_Carpool_pct',
    'PU_Transport_PublicTransit_pct', 'PU_Transport_Walked_pct',
    'PU_Transport_Bicycle_pct', 'PU_Transport_Other_pct',
    'PU_Transport_WorkFromHome_pct',
    'is_weekend', 'is_rush_hour', 'is_night', 'is_morning', 'is_afternoon',
    'hour', 'day_of_week', 'month',
]

CAT_FEATS_EMB = {
    'PULocationID':  265,
    'PU_Borough_id': 5,
}

CAT_ONEHOT = ['PU_Borough_id']

print(f'Zonas conservadas: {N_CLASSES} de 265')
print(f'Filas tras filtro: {len(df):,}')

## 2. Analisis del target

In [ ]:
top20 = df[TARGET].value_counts().head(20)
top20_names = []
for loc in top20.index:
    name = zone_lookup.loc[loc,'Zone'][:15] if loc in zone_lookup.index else str(loc)
    top20_names.append(f'{loc}\n{name}')

fig, axes = plt.subplots(1, 2, figsize=(15, 4))

axes[0].barh(top20_names[::-1], top20.values[::-1], color='steelblue')
axes[0].set_title('Top 20 zonas de destino')
axes[0].set_xlabel('N viajes')

BOROUGH_NAMES = {0:'Manhattan',1:'Brooklyn',2:'Queens',3:'Bronx',4:'Staten Island',5:'EWR',6:'Unknown'}
borough_dist  = df['DO_Borough_id'].value_counts().sort_index()
blabels = [BOROUGH_NAMES.get(i, str(i)) for i in borough_dist.index]
axes[1].bar(blabels, borough_dist.values, color=sns.color_palette('muted', len(borough_dist)))
axes[1].set_title('Destinos por borough')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout(); plt.show()
print(f'Top-1 zona = {top20.iloc[0]/len(df)*100:.1f}% de los viajes')
print(f'Top-20 zonas cubren el {top20.sum()/len(df)*100:.1f}% del total')

## 3. Split y preprocesamiento

In [ ]:
df_onehot = df.copy()
df_emb    = df.copy()
df_ae     = df.copy()

y = df['do_zone_idx'].values

idx = np.arange(len(df))
tv_idx, test_idx = train_test_split(idx, test_size=0.15,
                                     stratify=y, random_state=SEED)
train_idx, val_idx = train_test_split(tv_idx, test_size=0.1765,
                                       stratify=y[tv_idx], random_state=SEED)

print(f'Train: {len(train_idx):,} | Val: {len(val_idx):,} | Test: {len(test_idx):,}')

### 3.1 Preprocesamiento MLP base (one-hot)

In [ ]:
df_onehot_enc = pd.get_dummies(df_onehot, columns=CAT_ONEHOT, drop_first=False)
onehot_cols = [c for c in df_onehot_enc.columns
               if any(c.startswith(p+'_') for p in CAT_ONEHOT)]

features_base = NUM_FEATS + onehot_cols
X_base = df_onehot_enc[features_base].astype(np.float32).values
y_base = df_onehot_enc['do_zone_idx'].values

scaler_base = StandardScaler()
X_base[train_idx] = scaler_base.fit_transform(X_base[train_idx])
X_base[val_idx]   = scaler_base.transform(X_base[val_idx])
X_base[test_idx]  = scaler_base.transform(X_base[test_idx])

print(f'Features baseline: {X_base.shape[1]} | Clases: {N_CLASSES}')

### 3.2 DataLoaders baseline

In [ ]:
BATCH_SIZE = 256

def make_loader(X, y, idx, shuffle=True):
    Xt = torch.tensor(X[idx], dtype=torch.float32)
    yt = torch.tensor(y[idx], dtype=torch.long)
    return DataLoader(TensorDataset(Xt, yt), batch_size=BATCH_SIZE, shuffle=shuffle)

train_loader_base = make_loader(X_base, y_base, train_idx)
val_loader_base   = make_loader(X_base, y_base, val_idx,   shuffle=False)
test_loader_base  = make_loader(X_base, y_base, test_idx,  shuffle=False)

xb, yb = next(iter(train_loader_base))
print('Batch X:', xb.shape, '| Batch y:', yb.shape)

## 4. MLP Base (sin embeddings)

Variables categoricas de baja cardinalidad (PU_Borough_id) con one-hot.  
PULocationID se excluye para no agregar cientos de dimensiones -- entra como embedding en la seccion 5.  
Con muchas clases de destino, la metrica principal es **Top-5 Accuracy** ademas de accuracy estandar.

In [ ]:
class MLPBase(nn.Module):
    def __init__(self, input_dim, hidden_dims=(256, 128, 64), dropout=0.3, n_classes=N_CLASSES):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, n_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

model_base = MLPBase(input_dim=X_base.shape[1]).to(DEVICE)
print(model_base)
print(f'Parametros: {sum(p.numel() for p in model_base.parameters()):,}')

In [ ]:
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for Xb, yb in loader:
        Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(Xb)
        loss = criterion(logits, yb)
        loss.backward(); optimizer.step()
        total_loss += loss.item() * len(yb)
        correct += (logits.argmax(1) == yb).sum().item()
        total += len(yb)
    return total_loss / total, correct / total

@torch.no_grad()
def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    for Xb, yb in loader:
        Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
        logits = model(Xb)
        loss = criterion(logits, yb)
        total_loss += loss.item() * len(yb)
        correct += (logits.argmax(1) == yb).sum().item()
        total += len(yb)
    return total_loss / total, correct / total

@torch.no_grad()
def get_preds(model, loader):
    model.eval()
    y_true, y_pred, y_scores = [], [], []
    for Xb, yb in loader:
        logits = model(Xb.to(DEVICE))
        y_scores.append(torch.softmax(logits, dim=1).cpu().numpy())
        y_pred.extend(logits.argmax(1).cpu().numpy())
        y_true.extend(yb.numpy())
    return np.array(y_true), np.array(y_pred), np.vstack(y_scores)

def plot_curves(train_vals, val_vals, metric='Loss', title=''):
    plt.figure(figsize=(8, 4))
    plt.plot(train_vals, label='Train')
    plt.plot(val_vals,   label='Validacion')
    plt.xlabel('Epoca'); plt.ylabel(metric)
    plt.title(title or metric); plt.legend(); plt.tight_layout(); plt.show()

def eval_report(y_true, y_pred, y_scores, label=''):
    acc   = accuracy_score(y_true, y_pred)
    bacc  = balanced_accuracy_score(y_true, y_pred)
    top5  = top_k_accuracy_score(y_true, y_scores, k=5)
    top10 = top_k_accuracy_score(y_true, y_scores, k=10)
    print(f'=== {label} ===')
    print(f'  Accuracy:          {acc:.4f}')
    print(f'  Balanced Accuracy: {bacc:.4f}')
    print(f'  Top-5  Accuracy:   {top5:.4f}')
    print(f'  Top-10 Accuracy:   {top10:.4f}')
    return {'acc': acc, 'bacc': bacc, 'top5': top5, 'top10': top10}

In [ ]:
EPOCHS    = 60
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_base.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)

hist_base = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[]}

for epoch in range(1, EPOCHS + 1):
    tl, ta = train_epoch(model_base, train_loader_base, criterion, optimizer)
    vl, va = eval_epoch(model_base,  val_loader_base,   criterion)
    scheduler.step()
    hist_base['train_loss'].append(tl); hist_base['val_loss'].append(vl)
    hist_base['train_acc'].append(ta);  hist_base['val_acc'].append(va)
    if epoch % 10 == 0 or epoch == 1:
        print(f'Epoch {epoch:03d} | loss={tl:.4f} acc={ta:.3f} | val_loss={vl:.4f} val_acc={va:.3f}')

In [ ]:
plot_curves(hist_base['train_loss'], hist_base['val_loss'], 'Cross-Entropy Loss', 'MLP Base')
plot_curves(hist_base['train_acc'],  hist_base['val_acc'],  'Accuracy',           'MLP Base')

In [ ]:
y_true_base, y_pred_base, y_scores_base = get_preds(model_base, test_loader_base)
metrics_base = eval_report(y_true_base, y_pred_base, y_scores_base, 'MLP Base (test)')

from collections import Counter
errors = [(remap_inv[y_true_base[i]], remap_inv[y_pred_base[i]])
          for i in range(len(y_true_base)) if y_true_base[i] != y_pred_base[i]]
top_errors = Counter(errors).most_common(10)
print('\nTop 10 pares (real -> predicho) mas confundidos:')
for (real, pred), cnt in top_errors:
    rname = zone_lookup.loc[real,'Zone'] if real in zone_lookup.index else real
    pname = zone_lookup.loc[pred,'Zone'] if pred in zone_lookup.index else pred
    print(f'  {rname} -> {pname}: {cnt} veces')

## 5. MLP con Embeddings

Con muchas zonas de destino, la zona de origen (`PULocationID`) es la senal geoespacial mas informativa.  
El embedding le permite al modelo aprender que zonas geograficamente cercanas son similares en comportamiento,  
sin necesitar cientos de dimensiones one-hot.

In [ ]:
for col, n_cats in CAT_FEATS_EMB.items():
    df_emb[col] = df_emb[col].clip(0, n_cats - 1).astype(np.int64)
df_emb['PULocationID'] = (df_emb['PULocationID'] - 1).clip(0, 264).astype(np.int64)

X_num_emb = df_emb[NUM_FEATS].astype(np.float32).values
X_cat_emb = df_emb[list(CAT_FEATS_EMB.keys())].astype(np.int64).values
y_emb     = df_emb['do_zone_idx'].values

scaler_emb = StandardScaler()
X_num_emb[train_idx] = scaler_emb.fit_transform(X_num_emb[train_idx])
X_num_emb[val_idx]   = scaler_emb.transform(X_num_emb[val_idx])
X_num_emb[test_idx]  = scaler_emb.transform(X_num_emb[test_idx])

print(f'X_num: {X_num_emb.shape} | X_cat: {X_cat_emb.shape}')

In [ ]:
class EmbDataset(Dataset):
    def __init__(self, X_num, X_cat, y, idx):
        self.X_num = torch.tensor(X_num[idx], dtype=torch.float32)
        self.X_cat = torch.tensor(X_cat[idx], dtype=torch.long)
        self.y     = torch.tensor(y[idx],     dtype=torch.long)
    def __len__(self):  return len(self.y)
    def __getitem__(self, i): return self.X_num[i], self.X_cat[i], self.y[i]

train_ds_emb = EmbDataset(X_num_emb, X_cat_emb, y_emb, train_idx)
val_ds_emb   = EmbDataset(X_num_emb, X_cat_emb, y_emb, val_idx)
test_ds_emb  = EmbDataset(X_num_emb, X_cat_emb, y_emb, test_idx)

train_loader_emb = DataLoader(train_ds_emb, batch_size=BATCH_SIZE, shuffle=True)
val_loader_emb   = DataLoader(val_ds_emb,   batch_size=512, shuffle=False)
test_loader_emb  = DataLoader(test_ds_emb,  batch_size=512, shuffle=False)

In [ ]:
def emb_dim(n_cats):
    return min(50, (n_cats + 1) // 2)

class MLPWithEmb(nn.Module):
    def __init__(self, num_dim, cat_vocab, hidden_dims=(256,128,64), dropout=0.3, n_classes=N_CLASSES):
        super().__init__()
        self.embs = nn.ModuleList([nn.Embedding(n, emb_dim(n)) for n in cat_vocab])
        total_emb = sum(emb_dim(n) for n in cat_vocab)
        layers = []
        prev = num_dim + total_emb
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, n_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x_num, x_cat):
        emb_out = [emb(x_cat[:, i]) for i, emb in enumerate(self.embs)]
        x = torch.cat([x_num] + emb_out, dim=1)
        return self.net(x)

cat_vocabs = list(CAT_FEATS_EMB.values())
model_emb = MLPWithEmb(num_dim=X_num_emb.shape[1], cat_vocab=cat_vocabs).to(DEVICE)
print(model_emb)
print(f'Parametros: {sum(p.numel() for p in model_emb.parameters()):,}')

In [ ]:
def train_epoch_emb(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for xn, xc, yb in loader:
        xn, xc, yb = xn.to(DEVICE), xc.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(xn, xc)
        loss = criterion(logits, yb)
        loss.backward(); optimizer.step()
        total_loss += loss.item() * len(yb)
        correct += (logits.argmax(1) == yb).sum().item()
        total += len(yb)
    return total_loss / total, correct / total

@torch.no_grad()
def eval_epoch_emb(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    for xn, xc, yb in loader:
        xn, xc, yb = xn.to(DEVICE), xc.to(DEVICE), yb.to(DEVICE)
        logits = model(xn, xc)
        loss = criterion(logits, yb)
        total_loss += loss.item() * len(yb)
        correct += (logits.argmax(1) == yb).sum().item()
        total += len(yb)
    return total_loss / total, correct / total

@torch.no_grad()
def get_preds_emb(model, loader):
    model.eval()
    y_true, y_pred, y_scores = [], [], []
    for xn, xc, yb in loader:
        logits = model(xn.to(DEVICE), xc.to(DEVICE))
        y_scores.append(torch.softmax(logits, dim=1).cpu().numpy())
        y_pred.extend(logits.argmax(1).cpu().numpy())
        y_true.extend(yb.numpy())
    return np.array(y_true), np.array(y_pred), np.vstack(y_scores)

In [ ]:
EPOCHS_EMB    = 60
optimizer_emb = optim.Adam(model_emb.parameters(), lr=1e-3)
scheduler_emb = optim.lr_scheduler.StepLR(optimizer_emb, step_size=20, gamma=0.5)

hist_emb = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[]}

for epoch in range(1, EPOCHS_EMB + 1):
    tl, ta = train_epoch_emb(model_emb, train_loader_emb, criterion, optimizer_emb)
    vl, va = eval_epoch_emb(model_emb, val_loader_emb, criterion)
    scheduler_emb.step()
    hist_emb['train_loss'].append(tl); hist_emb['val_loss'].append(vl)
    hist_emb['train_acc'].append(ta);  hist_emb['val_acc'].append(va)
    if epoch % 10 == 0 or epoch == 1:
        print(f'Epoch {epoch:03d} | loss={tl:.4f} acc={ta:.3f} | val_loss={vl:.4f} val_acc={va:.3f}')

In [ ]:
plot_curves(hist_emb['train_loss'], hist_emb['val_loss'], 'Cross-Entropy Loss', 'MLP Embeddings')
plot_curves(hist_emb['train_acc'],  hist_emb['val_acc'],  'Accuracy',           'MLP Embeddings')

In [ ]:
y_true_emb, y_pred_emb, y_scores_emb = get_preds_emb(model_emb, test_loader_emb)
metrics_emb = eval_report(y_true_emb, y_pred_emb, y_scores_emb, 'MLP Embeddings (test)')

### 5.1 Visualizacion del espacio de embeddings de PULocationID

In [ ]:
from sklearn.decomposition import PCA

pu_weights = model_emb.embs[0].weight.data.cpu().numpy()
pca = PCA(n_components=2, random_state=SEED)
pu_2d = pca.fit_transform(pu_weights)

BOROUGH_MAP_COL = {'Manhattan': 0, 'Brooklyn': 1, 'Queens': 2, 'Bronx': 3, 'Staten Island': 4}
colors = zone_lookup['Borough'].map(BOROUGH_MAP_COL).reindex(range(1, 266)).fillna(-1).values

fig, ax = plt.subplots(figsize=(9, 7))
palette = sns.color_palette('muted', 5)
for bid, bname in {0:'Manhattan',1:'Brooklyn',2:'Queens',3:'Bronx',4:'Staten Island'}.items():
    mask = colors == bid
    ax.scatter(pu_2d[mask, 0], pu_2d[mask, 1], label=bname,
               color=palette[bid], alpha=0.75, s=45)

ax.set_title('PCA del embedding de PULocationID (265 zonas)')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax.legend(title='Borough de origen')
plt.tight_layout(); plt.show()

## 6. Analisis de sobreajuste y regularizacion

### 6.1 Analisis de curvas de entrenamiento

Las curvas de entrenamiento revelan un comportamiento inusual a primera vista: en el modelo con embeddings, la **loss de validacion es consistentemente menor que la de entrenamiento**, y la **accuracy de validacion supera a la de entrenamiento** durante casi todas las epocas. Esto no es overfitting -- es el efecto del **Dropout**, que apaga aleatoriamente neuronas durante el forward pass de entrenamiento pero esta inactivo en validacion. El modelo en validacion es efectivamente 'mas grande' que en entrenamiento, lo que explica la aparente inversion.

El modelo base muestra curvas train y val casi identicas y planas (~5.8% accuracy), lo que indica **underfitting**: la representacion one-hot sin PULocationID no captura suficiente estructura geoespacial para distinguir destinos. El modelo con embeddings supera claramente al base (~8% vs ~5.8%), confirmando que la representacion densa de las zonas de origen aporta informacion relevante.

Dado que no hay brecha creciente train/val, el problema actual no es overfitting sino capacidad de representacion. La regularizacion aqui cumple un rol preventivo: al aumentar la capacidad del modelo para buscar mejor performance, el riesgo de overfitting crece. A continuacion se entrena una variante con mayor capacidad y se evalua el efecto de dropout y weight decay.

### 6.2 MLP con embeddings regularizado (mayor capacidad + dropout aumentado)

In [ ]:
# Misma arquitectura y dropout que MLP Embeddings -- solo se agrega weight decay
# para aislar su efecto sobre el entrenamiento
model_reg = MLPWithEmb(
    num_dim=X_num_emb.shape[1],
    cat_vocab=cat_vocabs,
    hidden_dims=(256, 128, 64),   # identico a MLP Embeddings
    dropout=0.3,                  # identico a MLP Embeddings
    n_classes=N_CLASSES
).to(DEVICE)

optimizer_reg = optim.Adam(model_reg.parameters(), lr=1e-3, weight_decay=1e-3)  # unica diferencia
scheduler_reg = optim.lr_scheduler.StepLR(optimizer_reg, step_size=20, gamma=0.5)

hist_reg = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[]}

for epoch in range(1, EPOCHS_EMB + 1):
    tl, ta = train_epoch_emb(model_reg, train_loader_emb, criterion, optimizer_reg)
    vl, va = eval_epoch_emb(model_reg, val_loader_emb, criterion)
    scheduler_reg.step()
    hist_reg['train_loss'].append(tl); hist_reg['val_loss'].append(vl)
    hist_reg['train_acc'].append(ta);  hist_reg['val_acc'].append(va)
    if epoch % 10 == 0 or epoch == 1:
        print(f'Epoch {epoch:03d} | loss={tl:.4f} acc={ta:.3f} | val_loss={vl:.4f} val_acc={va:.3f}')

In [ ]:
# Comparacion de las tres curvas
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(hist_base['train_loss'], label='Base train',      lw=1.5, color='steelblue')
axes[0].plot(hist_base['val_loss'],   label='Base val',        lw=1.5, color='steelblue', ls='--')
axes[0].plot(hist_emb['train_loss'],  label='Emb train',       lw=1.5, color='coral')
axes[0].plot(hist_emb['val_loss'],    label='Emb val',         lw=1.5, color='coral',     ls='--')
axes[0].plot(hist_reg['train_loss'],  label='Emb+Reg train',   lw=1.5, color='green')
axes[0].plot(hist_reg['val_loss'],    label='Emb+Reg val',     lw=1.5, color='green',     ls='--')
axes[0].set_title('Cross-Entropy Loss'); axes[0].set_xlabel('Epoca'); axes[0].legend(fontsize=8)

axes[1].plot(hist_base['train_acc'],  label='Base train',      lw=1.5, color='steelblue')
axes[1].plot(hist_base['val_acc'],    label='Base val',        lw=1.5, color='steelblue', ls='--')
axes[1].plot(hist_emb['train_acc'],   label='Emb train',       lw=1.5, color='coral')
axes[1].plot(hist_emb['val_acc'],     label='Emb val',         lw=1.5, color='coral',     ls='--')
axes[1].plot(hist_reg['train_acc'],   label='Emb+Reg train',   lw=1.5, color='green')
axes[1].plot(hist_reg['val_acc'],     label='Emb+Reg val',     lw=1.5, color='green',     ls='--')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoca'); axes[1].legend(fontsize=8)

plt.suptitle('Comparacion train/val -- Base vs Embeddings vs Embeddings+Regularizacion')
plt.tight_layout(); plt.show()

In [ ]:
y_true_reg, y_pred_reg, y_scores_reg = get_preds_emb(model_reg, test_loader_emb)
metrics_reg = eval_report(y_true_reg, y_pred_reg, y_scores_reg, 'MLP Emb + Regularizacion (test)')

### 6.3 Discusion

El tercer modelo es identico al MLP con embeddings en arquitectura (hidden_dims, dropout) -- la unica diferencia es el **weight decay** (`1e-3` vs sin penalizacion). Esto permite aislar su efecto: si las curvas train/val se acercan o las metricas de test mejoran, el weight decay esta siendo util; si las metricas bajan, esta penalizando demasiado y habria que reducirlo.

El **weight decay** equivale a regularizacion L2 sobre los parametros del modelo: en cada paso de gradiente se empuja levemente los pesos hacia cero, desincentivando que el modelo asigne demasiada importancia a cualquier feature o conexion individual. Es especialmente util cuando hay muchas clases con pocos ejemplos, como en este problema donde las zonas menos frecuentes tienen muy pocas muestras de entrenamiento.

## 7. AutoEncoder + MLP

El AutoEncoder se entrena sobre las **variables numericas** (`NUM_FEATS`) para aprender una representacion comprimida. El encoder resultante se congela y su salida (el cuello de botella `z`) se usa como entrada a un MLP clasificador.

Ajusta `BOTTLENECK_DIM` para explorar el trade-off entre compresion e informacion retenida.

In [ ]:
# ── Hiperparametro principal: dimension del cuello de botella ─────────────────
BOTTLENECK_DIM = 12   # <-- cambia esto para experimentar (ej: 8, 16, 32, 64)

AE_EPOCHS  = 40
MLP_EPOCHS = 60
AE_LR      = 1e-3
MLP_LR     = 1e-3

### 7.1 Preparar datos (solo numericas)

In [ ]:
X_ae = df_ae[NUM_FEATS].astype(np.float32).values
y_ae = df_ae['do_zone_idx'].values

scaler_ae = StandardScaler()
X_ae[train_idx] = scaler_ae.fit_transform(X_ae[train_idx])
X_ae[val_idx]   = scaler_ae.transform(X_ae[val_idx])
X_ae[test_idx]  = scaler_ae.transform(X_ae[test_idx])

def make_loader_ae(X, y, idx, shuffle=True):
    Xt = torch.tensor(X[idx], dtype=torch.float32)
    yt = torch.tensor(y[idx], dtype=torch.long)
    return DataLoader(TensorDataset(Xt, yt), batch_size=BATCH_SIZE, shuffle=shuffle)

train_loader_ae = make_loader_ae(X_ae, y_ae, train_idx)
val_loader_ae   = make_loader_ae(X_ae, y_ae, val_idx,   shuffle=False)
test_loader_ae  = make_loader_ae(X_ae, y_ae, test_idx,  shuffle=False)

INPUT_DIM = X_ae.shape[1]
print(f'Input dim: {INPUT_DIM} | Bottleneck: {BOTTLENECK_DIM}')

### 7.2 Definir AutoEncoder

In [ ]:
class AutoEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dims=(128, 64), bottleneck=BOTTLENECK_DIM, dropout=0.2):
        super().__init__()
        # Encoder
        enc, prev = [], input_dim
        for h in hidden_dims:
            enc += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        enc.append(nn.Linear(prev, bottleneck))
        self.encoder = nn.Sequential(*enc)
        # Decoder (espejo del encoder)
        dec, prev = [], bottleneck
        for h in reversed(hidden_dims):
            dec += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        dec.append(nn.Linear(prev, input_dim))
        self.decoder = nn.Sequential(*dec)

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z

ae = AutoEncoder(input_dim=INPUT_DIM).to(DEVICE)
print(ae)
print(f'Parametros AE: {sum(p.numel() for p in ae.parameters()):,}')

### 7.3 Entrenar AutoEncoder

In [ ]:
ae_criterion  = nn.MSELoss()
ae_optimizer  = optim.Adam(ae.parameters(), lr=AE_LR)

hist_ae = {'train_loss': [], 'val_loss': []}

for epoch in range(1, AE_EPOCHS + 1):
    ae.train()
    batch_losses = []
    for Xb, _ in train_loader_ae:
        Xb = Xb.to(DEVICE)
        ae_optimizer.zero_grad()
        x_rec, _ = ae(Xb)
        loss = ae_criterion(x_rec, Xb)
        loss.backward(); ae_optimizer.step()
        batch_losses.append(loss.item())
    train_loss = np.mean(batch_losses)

    ae.eval()
    with torch.no_grad():
        val_losses = []
        for Xb, _ in val_loader_ae:
            Xb = Xb.to(DEVICE)
            x_rec, _ = ae(Xb)
            val_losses.append(ae_criterion(x_rec, Xb).item())
    val_loss = np.mean(val_losses)

    hist_ae['train_loss'].append(train_loss)
    hist_ae['val_loss'].append(val_loss)
    if epoch % 10 == 0 or epoch == 1:
        print(f'AE Epoch {epoch:03d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f}')

plot_curves(hist_ae['train_loss'], hist_ae['val_loss'], 'MSE Loss', f'AutoEncoder (bottleneck={BOTTLENECK_DIM})')

### 7.4 Congelar encoder y entrenar MLP clasificador

In [ ]:
# Congelar todos los parametros del AE
for param in ae.parameters():
    param.requires_grad = False

# MLP que recibe el bottleneck como entrada
mlp_ae = MLPBase(
    input_dim=BOTTLENECK_DIM,
    hidden_dims=(128, 64),
    dropout=0.3,
    n_classes=N_CLASSES
).to(DEVICE)

print(f'MLP sobre bottleneck -- input: {BOTTLENECK_DIM} dims')
print(f'Parametros entrenables: {sum(p.numel() for p in mlp_ae.parameters() if p.requires_grad):,}')

In [ ]:
def train_epoch_ae(encoder, mlp, loader, criterion, optimizer):
    mlp.train(); encoder.eval()
    total_loss, correct, total = 0, 0, 0
    for Xb, yb in loader:
        Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
        with torch.no_grad():
            z = encoder(Xb)
        optimizer.zero_grad()
        logits = mlp(z)
        loss = criterion(logits, yb)
        loss.backward(); optimizer.step()
        total_loss += loss.item() * len(yb)
        correct += (logits.argmax(1) == yb).sum().item()
        total += len(yb)
    return total_loss / total, correct / total

@torch.no_grad()
def eval_epoch_ae(encoder, mlp, loader, criterion):
    mlp.eval(); encoder.eval()
    total_loss, correct, total = 0, 0, 0
    for Xb, yb in loader:
        Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
        z = encoder(Xb)
        logits = mlp(z)
        loss = criterion(logits, yb)
        total_loss += loss.item() * len(yb)
        correct += (logits.argmax(1) == yb).sum().item()
        total += len(yb)
    return total_loss / total, correct / total

@torch.no_grad()
def get_preds_ae(encoder, mlp, loader):
    mlp.eval(); encoder.eval()
    y_true, y_pred, y_scores = [], [], []
    for Xb, yb in loader:
        Xb = Xb.to(DEVICE)
        z = encoder(Xb)
        logits = mlp(z)
        y_scores.append(torch.softmax(logits, dim=1).cpu().numpy())
        y_pred.extend(logits.argmax(1).cpu().numpy())
        y_true.extend(yb.numpy())
    return np.array(y_true), np.array(y_pred), np.vstack(y_scores)

In [ ]:
mlp_ae_optimizer = optim.Adam(mlp_ae.parameters(), lr=MLP_LR)
mlp_ae_scheduler = optim.lr_scheduler.StepLR(mlp_ae_optimizer, step_size=20, gamma=0.5)

hist_mlp_ae = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[]}

for epoch in range(1, MLP_EPOCHS + 1):
    tl, ta = train_epoch_ae(ae.encoder, mlp_ae, train_loader_ae, criterion, mlp_ae_optimizer)
    vl, va = eval_epoch_ae(ae.encoder,  mlp_ae, val_loader_ae,   criterion)
    mlp_ae_scheduler.step()
    hist_mlp_ae['train_loss'].append(tl); hist_mlp_ae['val_loss'].append(vl)
    hist_mlp_ae['train_acc'].append(ta);  hist_mlp_ae['val_acc'].append(va)
    if epoch % 10 == 0 or epoch == 1:
        print(f'Epoch {epoch:03d} | loss={tl:.4f} acc={ta:.3f} | val_loss={vl:.4f} val_acc={va:.3f}')

plot_curves(hist_mlp_ae['train_loss'], hist_mlp_ae['val_loss'], 'Cross-Entropy Loss',
           f'AE + MLP (bottleneck={BOTTLENECK_DIM})')
plot_curves(hist_mlp_ae['train_acc'],  hist_mlp_ae['val_acc'],  'Accuracy',
           f'AE + MLP (bottleneck={BOTTLENECK_DIM})')

In [ ]:
y_true_ae, y_pred_ae, y_scores_ae = get_preds_ae(ae.encoder, mlp_ae, test_loader_ae)
metrics_ae = eval_report(y_true_ae, y_pred_ae, y_scores_ae,
                          f'AE + MLP bottleneck={BOTTLENECK_DIM} (test)')

## 8. Analisis critico final

In [ ]:
results = pd.DataFrame({
    'Modelo':       ['MLP Base', 'MLP Embeddings', 'MLP Emb+Reg', f'AE+MLP (b={BOTTLENECK_DIM})'],
    'Accuracy':     [metrics_base['acc'],  metrics_emb['acc'],  metrics_reg['acc'],  metrics_ae['acc']],
    'Balanced Acc': [metrics_base['bacc'], metrics_emb['bacc'], metrics_reg['bacc'], metrics_ae['bacc']],
    'Top-5 Acc':    [metrics_base['top5'], metrics_emb['top5'], metrics_reg['top5'], metrics_ae['top5']],
    'Top-10 Acc':   [metrics_base['top10'],metrics_emb['top10'],metrics_reg['top10'],metrics_ae['top10']],
}).set_index('Modelo')
results.round(4)

### Puntos a desarrollar:

1. **Dificultad del problema**: con muchas clases y distribucion desbalanceada (Manhattan concentra la mayoria de destinos), la accuracy cruda puede ser enganosa -- analizar Top-K Accuracy y Balanced Accuracy.
2. **MLP base vs embeddings**: el embedding de PULocationID mejora la prediccion porque aprende similitudes entre zonas. Analizar para que zonas fue mas util.
3. **Efecto de la regularizacion**: comparar las tres curvas -- si el modelo regularizado reduce la brecha train/val sin perder performance en test, la regularizacion fue efectiva.
4. **Visualizacion de embeddings**: zonas del mismo borough quedan agrupadas en PCA? Hay estructura geografica emergente?
5. **AutoEncoder**: el bottleneck comprime informacion util para clasificacion, o pierde senal importante?